In [1]:
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import numpy as np

sys.path.insert(0,"/software/local/languages/miniforge3/envs/elena/lib/python3.12/site-packages/")

sys.path.insert(0,"/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12/site-packages")

print(sys.path)

import torch
import os
import pickle
import einops

#import plenoptic as po

from model.layers.encoder import *
from model.layers.decoder import *
from model.layers.processor import *
from model.layers.graph_net_block import *
from model.data.dataloader_graphnet import *
from model.data.load_data import *
from model.forecast import GraphSatelliteForecaster
from model.loss_functions import *
from model.evaluation import *

import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
import time
from datetime import datetime
import json
import argparse

import random

%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import timeit

['/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12/site-packages', '/software/local/languages/miniforge3/envs/elena/lib/python3.12/site-packages/', '/user/work/ef17148/GCN/graphnet/graphnet_LPDM_emulator', '/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python312.zip', '/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12', '/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12/lib-dynload', '', '/user/home/ef17148/.local/lib/python3.12/site-packages', '__editable__.openghg_inversions-0.2.0.finder.__path_hook__', '/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12/site-packages']


# Loading the data

The base model just loads all the data from the passed/existing paths, without cropping or applying transforms. use it as a parent class to build specific data objects

### Base class

In [2]:
original_data = LoadBaseSatelliteData(year="2016", region="BRAZIL", month="06", freq=40, verbose=True, load_everything=True)

---- LOADING FOOTPRINTS
Loading footprint data from /group/chemistry/acrg/LPDM/fp_NAME_pre20210701/SOUTHAMERICA/*BRAZIL*SOUTHAMERICA_201606*.nc
reduced the number of datapoints by frequency 40
Loading 48 footprints

 ---- LOADING MET
Loading meteorology from /group/chemistry/acrg/met_archive/UM/SOUTHAMERICA/SOUTHAMERICA_Met_201606*.nc


/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/user/work/ef17148/oldstuff/ef17148/.conda/envs/new_graphnet/lib/python3.12/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True})


---- LOADING TOPOG
trying to load topography from /group/chemistry/acrg/LPDM/topog_NAME/TopogUMG_Mk8_global.nc
---- All done!


In [4]:
original_data.fp_data_full # the footprint file as loaded

<xarray.Dataset>
Dimensions:               (time: 48, lon: 190, lat: 357, lev: 1, height: 20)
Coordinates:
  * time                  (time) datetime64[ns] 2016-06-01T16:03:06 ... 2016-...
  * lon                   (lon) float32 -91.33 -90.98 -90.62 ... -25.15 -24.8
  * lat                   (lat) float32 -60.98 -60.75 -60.51 ... 22.09 22.32
  * lev                   (lev) |S1 b'c'
  * height                (height) float32 500.0 1.5e+03 ... 1.85e+04 1.95e+04
Data variables:
    fp                    (lat, lon, time) float32 dask.array<chunksize=(357, 190, 1), meta=np.ndarray>
    temperature           (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    pressure              (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    wind_speed            (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    wind_direction        (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    PBLH                  (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    release_lon           (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    release_lat           (time) float32 dask.array<chunksize=(1,), meta=np.ndarray>
    particle_locations_n  (height, lon, time) float32 dask.array<chunksize=(20, 190, 1), meta=np.ndarray>
    particle_locations_e  (height, lat, time) float32 dask.array<chunksize=(20, 357, 1), meta=np.ndarray>
    particle_locations_s  (height, lon, time) float32 dask.array<chunksize=(20, 190, 1), meta=np.ndarray>
    particle_locations_w  (height, lat, time) float32 dask.array<chunksize=(20, 357, 1), meta=np.ndarray>
Attributes:
    max_level:  17
    author:     rt17603
    created:    2019-06-01 12:37:14.871106

In [5]:
original_data.met_file # the met file as loaded

<xarray.Dataset>
Dimensions:                              (lat: 357, levels: 20, lon: 239,
                                          time: 720)
Coordinates:
  * lat                                  (lat) float64 -60.98 -60.75 ... 22.32
    sigma                                (levels) float32 dask.array<chunksize=(20,), meta=np.ndarray>
    level_height_0                       (levels) float32 dask.array<chunksize=(20,), meta=np.ndarray>
    sigma_0                              (levels) float32 dask.array<chunksize=(20,), meta=np.ndarray>
  * levels                               (levels) int32 1 3 6 9 ... 48 51 54 57
    level_height                         (levels) float32 dask.array<chunksize=(20,), meta=np.ndarray>
  * lon                                  (lon) float64 -91.33 -90.98 ... -7.553
  * time                                 (time) datetime64[ns] 2016-06-01 ......
Data variables:
    air_pressure                         (levels, lat, lon, time) float32 dask.array<chunksize=(20, 357, 239, 500), meta=np.ndarray>
    air_pressure_at_sea_level            (lat, lon, time) float32 dask.array<chunksize=(357, 239, 500), meta=np.ndarray>
    air_temperature                      (levels, lat, lon, time) float32 dask.array<chunksize=(20, 357, 239, 500), meta=np.ndarray>
    atmosphere_boundary_layer_thickness  (lat, lon, time) float32 dask.array<chunksize=(357, 239, 500), meta=np.ndarray>
    surface_air_pressure                 (lat, lon, time) float32 dask.array<chunksize=(357, 239, 500), meta=np.ndarray>
    surface_upward_sensible_heat_flux    (lat, lon, time) float32 dask.array<chunksize=(357, 239, 500), meta=np.ndarray>
    upward_air_velocity                  (levels, lat, lon, time) float32 dask.array<chunksize=(20, 357, 239, 500), meta=np.ndarray>
    x_wind                               (levels, lat, lon, time) float32 dask.array<chunksize=(20, 357, 239, 500), meta=np.ndarray>
    y_wind                               (levels, lat, lon, time) float32 dask.array<chunksize=(20, 357, 239, 500), meta=np.ndarray>
Attributes:
    source:           Data from Met Office Unified Model
    author:           Elena Fillola (ef17148)
    created:          2022-11-14 13:11
    transformations:  interpolated linearly in space to NAME resolution, inte...

### Square class

the square class cuts the data to a square of size size x size around the measurement coordinates, with the meteorology and topography also aligned to this format

In [9]:
cropped_data = LoadSquareSatelliteData(year=2016, region="SAHARA", freq=40,  size=50, verbose=True, load_everything=True)

---- LOADING FOOTPRINTS
Loading footprint data from /group/chemistry/acrg/LPDM/fp_NAME_pre20210701/NORTHAFRICA/*SAHARA*NORTHAFRICA_2016*.nc
there was an error opening the dataset. checking if any of the files are in the bad files list
at least one of the files was in the bad files list, opening with workaround
reduced the number of datapoints by frequency 40
Loading 323 footprints
----- Cutting footprints to square of size 50
careful! We had to pad the footprints along the latitude dimension to extract size 50 (0 and 11 idxs on either side) Padding with nan
23 footprints were at least partially filled with nans because they were cutting outside of the footprint file domain (this is 7.12% of samples)
Padding was needed for the following number of footprints along each direction: {'N': 23, 'S': 0, 'E': 0, 'W': 0}

 ---- LOADING MET
Loading meteorology from /group/chemistry/acrg/met_archive/UM/NORTHAFRICA/NORTHAFRICA_Met_2016*.nc
----- Cutting met

---- LOADING TOPOG
trying to load topogr

In [16]:
cropped_data.met # met object, interpolated to the same time as the footprints and cropped to the right size

<xarray.Dataset>
Dimensions:                              (levels: 20, lat: 50, lon: 50,
                                          time: 323, time_delta: 1)
Coordinates:
  * levels                               (levels) int32 1 3 6 9 ... 48 51 54 57
    sigma                                (levels) float32 dask.array<chunksize=(20,), meta=np.ndarray>
    level_height                         (levels) float32 dask.array<chunksize=(20,), meta=np.ndarray>
  * time                                 (time) datetime64[ns] 2016-01-01T11:...
  * time_delta                           (time_delta) int64 0
  * lat                                  (lat) int64 0 1 2 3 4 ... 46 47 48 49
  * lon                                  (lon) int64 0 1 2 3 4 ... 46 47 48 49
Data variables: (12/13)
    air_pressure                         (levels, lat, lon, time) float64 dask.array<chunksize=(20, 50, 50, 1), meta=np.ndarray>
    air_pressure_at_sea_level            (lat, lon, time) float64 dask.array<chunksize=(50, 50, 1), meta=np.ndarray>
    air_temperature                      (levels, lat, lon, time) float64 dask.array<chunksize=(20, 50, 50, 1), meta=np.ndarray>
    atmosphere_boundary_layer_thickness  (lat, lon, time) float64 dask.array<chunksize=(50, 50, 1), meta=np.ndarray>
    surface_air_pressure                 (lat, lon, time) float64 dask.array<chunksize=(50, 50, 1), meta=np.ndarray>
    upward_air_velocity                  (levels, lat, lon, time) float64 dask.array<chunksize=(20, 50, 50, 1), meta=np.ndarray>
    ...                                   ...
    y_wind                               (levels, lat, lon, time) float64 dask.array<chunksize=(20, 50, 50, 1), meta=np.ndarray>
    fp_time                              (time) datetime64[ns] 2016-01-01T11:...
    lat_coords                           (time, lat) float64 31.57 ... 21.27
    lon_coords                           (time, lon) float64 20.4 ... 38.35
    wind_angle                           (levels, lat, lon, time) float64 dask.array<chunksize=(20, 50, 50, 1), meta=np.ndarray>
    wind_speed                           (levels, lat, lon, time) float64 dask.array<chunksize=(20, 50, 50, 1), meta=np.ndarray>
Attributes:
    original_met_attrs:  {'author': 'Elena Fillola (ef17148)', 'created': '20...

# Setting up the data

The variables to extract should be passed as follows:
- met_variables: a dict of meteorological variables, of format {"var_name":[list of levels], "surface_var_name":[]}
- static_variables: a list of static variables to add. see `get_static_variables_functions` for a list of valid parameters, or add your own

In [24]:
variables = {"x_wind":[3,15], "y_wind":[3], "surface_air_pressure":[]}
static_variables=["lat_coords", "lon_coords", "x_coords", "y_coords", "topog", "landcover"]
inputs, input_names = get_square_satellite_inputs(cropped_data, variables, static_variables=static_variables, time_deltas=[24], return_variable_names=True)

---Preparing met
extracting met at t-H for H in: [0, 24]
Setting up variables with levels: ['x_wind', 'y_wind']
Setting up surface variables: ['surface_air_pressure']
Setting up static variables: ['lat_coords', 'lon_coords', 'x_coords', 'y_coords', 'topog', 'landcover']
(2, 50, 50)


In [28]:
inputs # the inputs are a stacked DataArray, of dimensions fp_time, lat, lon, and variable
# the inputs are lazy loaded, so they're not actually in memory until you run inputs.load(), 
# or until you attempt to use or extract the values

<xarray.DataArray 'stacked_levels_met' (fp_time: 322, lat: 50, lon: 50,
                                        variable_name: 14)>
dask.array<concatenate, shape=(322, 50, 50, 14), dtype=float64, chunksize=(1, 50, 50, 6), chunktype=numpy.ndarray>
Coordinates:
  * fp_time        (fp_time) datetime64[ns] 2016-01-02T12:04:33.500000 ... 20...
  * lat            (lat) int64 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * lon            (lon) int64 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * variable_name  (variable_name) object MultiIndex
Attributes:
    standard_name:  x_wind
    units:          m s-1
    source:         Data from Met Office Unified Model
    STASH:          [1 0 2]

In [26]:
inputs.variable_name # contains tuples specific to each 2D variable, specifying (variable, level, time_delta)
# surface variables always have level=0, and static variables have level=0 and time_delta=0

<xarray.DataArray 'variable_name' (variable_name: 14)>
array([('y_wind', 3, 0), ('y_wind', 3, 24), ('x_wind', 3, 0),
       ('x_wind', 15, 0), ('x_wind', 3, 24), ('x_wind', 15, 24),
       ('surface_air_pressure', 0, 0), ('surface_air_pressure', 0, 24),
       ('lat_coords', 0, 0), ('lon_coords', 0, 0), ('y_coords', 0, 0),
       ('x_coords', 0, 0), ('topog', 0, 0), ('landcover', 0, 0)], dtype=object)
Coordinates:
  * variable_name  (variable_name) object MultiIndex

In [27]:
input_names # input_names is a dict with the same vaues as inputs.variable_name, but it's easier to search

[{'var': 'y_wind', 'level': 3, 'time_delta': 0, 'type': 'met'},
 {'var': 'y_wind', 'level': 3, 'time_delta': 24, 'type': 'met'},
 {'var': 'x_wind', 'level': 3, 'time_delta': 0, 'type': 'met'},
 {'var': 'x_wind', 'level': 15, 'time_delta': 0, 'type': 'met'},
 {'var': 'x_wind', 'level': 3, 'time_delta': 24, 'type': 'met'},
 {'var': 'x_wind', 'level': 15, 'time_delta': 24, 'type': 'met'},
 {'var': 'surface_air_pressure', 'time_delta': 0, 'type': 'surface_met'},
 {'var': 'surface_air_pressure', 'time_delta': 24, 'type': 'surface_met'},
 {'var': 'lat_coords', 'type': 'static'},
 {'var': 'lon_coords', 'type': 'static'},
 {'var': 'y_coords', 'type': 'static'},
 {'var': 'x_coords', 'type': 'static'},
 {'var': 'topog', 'type': 'static'},
 {'var': 'landcover', 'type': 'static'}]

In [ ]:
variables = {"x_wind":[3,15], "y_wind":[3], "surface_air_pressure":[]}
static_variables=["lat_coords", "lon_coords", "x_coords", "y_coords", "topog", "landcover"]
inputs_array, input_names = get_square_satellite_inputs(cropped_data, variables, static_variables=static_variables, time_deltas=[24], return_variable_names=True, return_asarray=True)
# passing return_asarray=True loads the inputs into memory and returns them with the shape (time, lat*lon, variables). might take a while!

In [ ]:
# passing return_asarray=True loads the inputs into memory and returns them with the shape (time, lat*lon, variables)
# this is what the transform function is expecting, but could be improved or avoided!
np.shape(inputs_array)

### Preparing the dataset
FootprintsDataset applies any necessary transforms on the data. the functions used to transform the data are saved in the dataset object. they can be passed to another instance FootprintsDataset (using `test_mode=train_dataset.transform_parameters`), to apply the known transforms to an unseen dataset

In [ ]:
dataset = FootprintsDataset(inputs=inputs, fp=cropped_data.fp_data, input_names=input_names, input_transforms=["clever_transform_3"], output_transforms= ["logv4"])